In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
boston = pd.read_csv(r"C:\Users\gouth\Desktop\Goutham\Stats\DataSet\Boston_Housing Data.csv")
boston

In [ ]:
#Metropolis–Hastings
X = boston[['RM']]
y = boston['MEDV']
X = (X - X.mean()) / X.std()
def log_likelihood(beta, X, y, sigma=1.0):
    y_pred = beta[0] + beta[1] * X.squeeze()
    return np.sum(norm.logpdf(y, loc=y_pred, scale=sigma))
def log_prior(beta):
    return np.sum(norm.logpdf(beta, loc=0, scale=10))
def log_posterior(beta, X, y, sigma=1.0):
    return log_prior(beta) + log_likelihood(beta, X, y, sigma)
np.random.seed(42)
n_iter = 5000
beta_samples = np.zeros((n_iter, 2))
beta_current = np.array([0.0, 0.0])
for i in range(n_iter):
    beta_proposal = beta_current + np.random.normal(0, 0.1, size=2)
    log_accept_ratio = log_posterior(beta_proposal, X, y) - log_posterior(beta_current, X, y)
    if np.log(np.random.rand()) < log_accept_ratio:
        beta_current = beta_proposal
    beta_samples[i] = beta_current  # Always store current state
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(beta_samples[:,0])
plt.title("Trace of β0 (Intercept)")
plt.subplot(1,2,2)
plt.plot(beta_samples[:,1])
plt.title("Trace of β1 (Slope)")
plt.show()

In [ ]:
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# Example: X = standardized predictor (e.g. RM), y = target (e.g. MEDV)
# Make sure you already defined X and y before running this block

with pm.Model() as model:
    # Priors
    beta0 = pm.Normal("beta0", mu=0, sigma=10)
    beta1 = pm.Normal("beta1", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # Linear regression model
    mu = beta0 + beta1 * X.squeeze()

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)

    # Sampling (PyMC uses NUTS by default, which is Gibbs-like for continuous variables)
    trace = pm.sample(2000, tune=1000, cores=2, return_inferencedata=True)

# Posterior summary
print(az.summary(trace, var_names=["beta0", "beta1", "sigma"]))

# Trace plots
az.plot_trace(trace, var_names=["beta0", "beta1", "sigma"])
plt.show()

# Posterior distributions
az.plot_posterior(trace, var_names=["beta0", "beta1", "sigma"])
plt.show()